# Demo del agente: video de la defensa

Este notebook es solo para grabar el video. Hace cinco preguntas al agente y muestra,
en cada una, qué herramienta eligió. Cada pregunta va en una conversación nueva. El trabajo completo está en `obligatorio_agentic_rag.ipynb`.

| Sección del notebook principal | Qué muestra |
|---|---|
| 1. Preparación | Configuración, clave de Groq y bases disponibles |
| 2. Dataset | Los 6 CSV unidos y sin duplicados: 82.486 correos |
| 3. Chunking y embeddings | Fragmentos de 800 caracteres y similitud con e5-small |
| 4. Base vectorial | ChromaDB: correos y papers, búsqueda semántica y umbral |
| 5. Herramientas | Las seis tools que puede llamar el agente |
| 6. Agente con LangGraph | El grafo ReAct y el prompt de sistema |
| 7. Con y sin contexto | La misma pregunta con RAG y sin RAG |
| 8. Pruebas del chatbot | 12 casos y si eligió la tool correcta |
| 9. Memoria y sesiones | Corto y largo plazo, retomar y usuarios en paralelo |
| 10. Chatbot | El chat por consola con sus comandos |
| 11. Limitaciones | Lo que el agente todavía no resuelve |
| 12. Experimentos | Chunking, recuperación, embeddings, API y comparación de LLMs |

In [1]:
import os
import sys
import tempfile
from pathlib import Path

# Salida limpia para el video: sin barras de progreso ni avisos de Hugging Face.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

import transformers
from IPython.display import Markdown, display

transformers.logging.disable_progress_bar()
sys.path.insert(0, str(Path.cwd()))

from src import agente as modulo_agente
from src import chatbot, config, memoria, recuperacion

# Memoria temporal: la demo arranca sin recuerdos y no toca memoria.sqlite3.
config.ARCHIVO_MEMORIA = Path(tempfile.mkdtemp()) / "memoria_demo.sqlite3"
agente = modulo_agente.construir_agente(memoria.crear_memoria_corto_plazo(), memoria.crear_memoria_largo_plazo())
USUARIO = "nico"

# Carga ahora el modelo de embeddings, los índices y el dataset, para que no aparezca en las respuestas.
recuperacion._indice(config.COLECCION_PAPERS)
recuperacion._indice(config.COLECCION_DATASET)
recuperacion._tabla_dataset()


def preguntar(texto, conversacion):
    """Envía un mensaje al agente y muestra la respuesta junto con las tools que usó."""
    # Cada pregunta va en su propia conversación (thread_id), como en el chat con /nueva.
    sesion = {"configurable": {"thread_id": f"{USUARIO}-{conversacion}", "user_id": USUARIO}}
    mensajes = agente.invoke({"messages": [("user", texto)]}, sesion)["messages"]
    tools = chatbot._herramientas_usadas(mensajes)
    display(Markdown(f"**Usuario (conversación {conversacion}):** {texto}\n\n"
                     f"**Tools usadas:** `{', '.join(tools) or 'ninguna'}`\n\n"
                     f"**Agente:** {mensajes[-1].content}"))


print("Agente listo")

Agente listo


## 1. Pregunta sobre un paper
El agente busca en los papers indexados en ChromaDB.

In [2]:
preguntar("¿Cuántos parámetros tiene GPT-3?", conversacion=1)

**Usuario (conversación 1):** ¿Cuántos parámetros tiene GPT-3?

**Tools usadas:** `buscar_en_papers`

**Agente:** GPT-3 tiene **175 mil millones (175B) de parámetros** (Language Models are Few-Shot Learners, pág. 40).

El paper también menciona que entrenaron una serie de modelos más pequeños, de 125 millones a 13 mil millones de parámetros, para comparar su rendimiento con el de GPT-3 (pág. 6).

## 2. Estadística exacta del dataset
Para contar no sirve la búsqueda semántica: usa pandas sobre los 82.486 correos.

In [3]:
preguntar("¿Cuántos correos de phishing hay en el dataset?", conversacion=2)

**Usuario (conversación 2):** ¿Cuántos correos de phishing hay en el dataset?

**Tools usadas:** `estadisticas_dataset`

**Agente:** En el dataset hay **42.891 correos de phishing** (de un total de 82.486; los otros 39.595 son legítimos).

Desglose por corpus de origen:
- CEAS_08: 21.842
- Enron: 13.976
- SpamAssasin: 1.718
- Nigerian_Fraud: 3.332
- Nazario: 1.565
- Ling: 458

## 3. Pregunta sin respuesta en los datos
Ningún fragmento supera el umbral de relevancia, así que no inventa.

In [4]:
preguntar("¿Qué dice el paper de Attention sobre el mundial de fútbol 2022?", conversacion=3)

**Usuario (conversación 3):** ¿Qué dice el paper de Attention sobre el mundial de fútbol 2022?

**Tools usadas:** `buscar_en_papers`

**Agente:** No tengo información sobre el mundial de fútbol 2022 en el paper de Attention: los fragmentos recuperados no mencionan nada sobre fútbol. Ese paper se enfoca en el mecanismo de atención (attention) para modelos de lenguaje, no en temas deportivos.

Si querés, puedo buscar en el paper temas relacionados con su contenido real (por ejemplo, cómo funciona la atención, la arquitectura Transformer, etc.).

## 4. Guardar un recuerdo
El agente decide guardar el dato en la memoria de largo plazo, asociada al usuario.

In [5]:
preguntar("Hola! Soy Nico y trabajo en ciberseguridad.", conversacion=4)
print("Recuerdos guardados:", memoria.leer_recuerdos(agente.store, USUARIO))

**Usuario (conversación 4):** Hola! Soy Nico y trabajo en ciberseguridad.

**Tools usadas:** `guardar_recuerdo`

**Agente:** ¡Hola, Nico! Qué gusto saludarte. ¿En qué te puedo ayudar hoy? Puedo consultarte sobre los papers cargados (Attention, GPT, GPT-3, Scaling Laws, Subliminal Learning, DDIA) o sobre el dataset de correos de phishing y legítimos.

Recuerdos guardados: ['Se llama Nico y trabaja en ciberseguridad']


## 5. Conversación nueva, mismo usuario
Otro `thread_id`, sin historial. Lo que sabe de Nico viene de la memoria de largo plazo.

In [6]:
preguntar("¿Te acordás cómo me llamo y a qué me dedico?", conversacion=5)

**Usuario (conversación 5):** ¿Te acordás cómo me llamo y a qué me dedico?

**Tools usadas:** `ninguna`

**Agente:** Sí, me acordás: te llamás Nico y trabajás en ciberseguridad. ¿En qué te puedo ayudar hoy?